# Demanda y dotación de call center: Preparación y controles de datos

Ejecuta la etapa anterior primero. Las cifras y conclusiones siguientes se calculan con los datos de esta ejecución; no edites manualmente sus valores. Consulta [metodología](../docs/metodologia.md) para interpretar resultados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("call_center_staffing")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from call_center_staffing.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from call_center_staffing.analysis import run_stage
import pandas as pd
metrics, comment = run_stage(PROJECT_ROOT, "preparacion")
display(Markdown(comment))
scalars = {k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}
if scalars:
    display(pd.DataFrame(list(scalars.items()), columns=["Indicador", "Valor"]))
if "candidatos" in metrics:
    display(pd.DataFrame(metrics["candidatos"]))
if "baseline" in metrics and "modelo" in metrics:
    comparison = {key: {k: v for k, v in metrics[key].items() if not isinstance(v, (dict, list))} for key in ["baseline", "modelo"]}
    display(pd.DataFrame(comparison).T)
if "clasificacion" in metrics.get("modelo", {}):
    display(pd.DataFrame(metrics["modelo"]["clasificacion"]).T)
for image_name in ["eda.png", "evaluacion.png"]:
    image_path = PROJECT_ROOT / "reports/preparacion" / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))


# Demanda y dotación de call center: preparacion

Se agregan **5,000 registros de Kaggle** en **1,800 intervalos de 30 minutos**. Se incluyen llamadas no atendidas y ceros en el horario operativo configurado, cuya cobertura se asume completa. Split cronológico por días 60/20/20. AHT estimado exclusivamente en train: **256.8 segundos**, conversación atendida más **30 segundos supuestos de trabajo posterior**. La fuente no acredita cobertura de una operación real completa; no se generan llamadas sintéticas adicionales.


,Indicador,Valor
0,fuente,https://www.kaggle.com/datasets/akash1vishwaka...
1,sha256,8d57d20fff56e84c23de6fce05bc31fb23d932ed00c3ee...
2,llamadas,5000
3,intervalos,1800
4,intervalo_minutos,30
5,horario_asumido,09:00–19:00 todos los días
6,fecha_min,2021-01-01
7,fecha_max,2021-03-31
8,aht_estimado_solo_train_segundos,256.842126
9,wrapup_asumido_segundos,30


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/01_preparacion_datos.json")
from call_center_staffing.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))


[Abrir informe integral](../reports/informe_final/informe_final.md)